<a href="https://colab.research.google.com/github/BraveKing15/415-Final-Project/blob/Section-4.1/Section4_2_Q1_IoU_Evaluation_(5).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 2 — Section 4.2, Question 1: Quantitative IoU Evaluation

We run the detection pipeline (multi-scale sliding window + NMS from Section 4.1) across the full Stanford Dogs Dataset and report IoU metrics.

Results are cached to Google Drive so the heavy inference step never needs to be re-run.

## Imports

In [ ]:
import os, json
import xml.etree.ElementTree as ET

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms

## Setup: Dataset + Model

Extracts the Stanford Dogs Dataset from tarballs on Google Drive.  
Skip extraction if you already ran Section 4.1 in the same Colab session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT = '/content/drive/MyDrive/Colab Notebooks/project'
DATASET_DIR = f'{DRIVE_PROJECT}/Part2/Stanford Dog Dataset'
MODEL_PATH = f'{DRIVE_PROJECT}/Part1/resnet50_cats_dogs.pth'
CACHE_PATH = f'{DRIVE_PROJECT}/Part2/detections_cache.json'

print('Extracting images...')
!tar -xf "{DATASET_DIR}/images.tar" -C /content/

print('Extracting annotations...')
!tar -xf "{DATASET_DIR}/annotation.tar" -C /content/
print('Done.')

IMG_DIR = '/content/Images'
ANN_DIR = '/content/Annotation'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, 2)  # cat=0, dog=1
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model = model.to(device)
model.eval()
print('Model loaded on', device)

## Preprocessing

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

## Helper Functions (same as Section 4.1)

In [ ]:
def parse_annotation(annotation_path):
    tree = ET.parse(annotation_path)
    root = tree.getroot()
    boxes = []
    for obj in root.findall('object'):
        bb = obj.find('bndbox')
        boxes.append((int(bb.find('xmin').text), int(bb.find('ymin').text), int(bb.find('xmax').text), int(bb.find('ymax').text)))
    return boxes

def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return inter / (areaA + areaB - inter + 1e-6)

def nms(detections, iou_threshold=0.3):
    if not detections:
        return []
    detections = sorted(detections, key=lambda d: d[4], reverse=True)
    kept = []
    while detections:
        best = detections.pop(0)
        kept.append(best)
        detections = [d for d in detections if compute_iou(best[:4], d[:4]) < iou_threshold]
    return kept

def sliding_window(image, model, device, preprocess, scales=None, stride=64, threshold=0.80, batch_size=64, max_size=640):
    #Multi-scale sliding window. Image is capped at max_size px on its longest side before sliding; detected coordinates are scaled back to original space

    if scales is None:
        scales = [0.2, 0.3, 0.5, 0.75, 1.0]

    W, H = image.size
    rs = min(max_size / max(W, H), 1.0)   # resize scale (only downscale)
    if rs < 1.0:
        image = image.resize((int(W * rs), int(H * rs)), Image.BILINEAR)
    W2, H2 = image.size

    detections = []
    for scale in scales:
        win_size = int(min(W2, H2) * scale)
        if win_size < 32:
            continue
        crops, coords = [], []
        ys = list(range(0, H2 - win_size + 1, stride))
        xs = list(range(0, W2 - win_size + 1, stride))
        if not ys or not xs:
            continue
        if ys[-1] < H2 - win_size: ys.append(H2 - win_size)
        if xs[-1] < W2 - win_size: xs.append(W2 - win_size)
        for y in ys:
            for x in xs:
                crop = image.crop((x, y, x + win_size, y + win_size))
                crops.append(preprocess(crop))
                coords.append((x, y, x + win_size, y + win_size))
        if not crops:
            continue
        all_probs = []
        for i in range(0, len(crops), batch_size):
            batch = torch.stack(crops[i:i + batch_size]).to(device)
            with torch.no_grad():
                all_probs.append(torch.softmax(model(batch), dim=1)[:, 1].cpu().numpy())
        probs = np.concatenate(all_probs)
        for (x1, y1, x2, y2), score in zip(coords, probs):
            if score >= threshold:
                #scale coordinates back to original image space
                detections.append((int(x1 / rs), int(y1 / rs), int(x2 / rs), int(y2 / rs), float(score)))
    return detections
    #Returns list of (x1, y1, x2, y2, score)

## Build Image–Annotation Pairs

In [ ]:
pairs = []
for breed_folder in sorted(os.listdir(IMG_DIR)):
    img_breed_dir = os.path.join(IMG_DIR, breed_folder)
    ann_breed_dir = os.path.join(ANN_DIR, breed_folder)
    if not os.path.isdir(img_breed_dir) or not os.path.isdir(ann_breed_dir):
        continue
    for fname in sorted(os.listdir(img_breed_dir)):
        if not fname.lower().endswith('.jpg'):
            continue
        img_id = fname.replace('.jpg', '')
        ann_path = os.path.join(ann_breed_dir, img_id)
        if os.path.exists(ann_path):
            pairs.append((os.path.join(img_breed_dir, fname), ann_path))

print(f'Total annotated images: {len(pairs)} across {len(os.listdir(IMG_DIR))} breeds')

## Evaluation with Cache + Resume

Results are saved to `detections_cache.json` on Drive every 50 images.  
If Colab disconnects, re-running this cell skips already-processed images and continues where it left off.

In [ ]:
# Load existing cache if present
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH) as f:
        results = json.load(f)
    print(f'Loaded {len(results)} cached results.')
else:
    results = []

processed_ids = {r['img_id'] for r in results}
todo = [(img, ann) for img, ann in pairs
        if os.path.basename(img).replace('.jpg', '') not in processed_ids]
print(f'Remaining: {len(todo)} / {len(pairs)}')

for idx, (img_path, ann_path) in enumerate(todo):
    img_id = os.path.basename(img_path).replace('.jpg', '')
    breed = os.path.basename(os.path.dirname(img_path))
    image = Image.open(img_path).convert('RGB')
    gt_boxes = parse_annotation(ann_path)

    final_dets = nms(sliding_window(image, model, device, preprocess))

    if final_dets and gt_boxes:
        iou = max(compute_iou(final_dets[0][:4], gt) for gt in gt_boxes)
        detected = True
    else:
        iou = 0.0
        detected = False

    results.append({'img_id': img_id, 'breed': breed, 'detected': detected, 'iou': iou})

    if (idx + 1) % 50 == 0:
        with open(CACHE_PATH, 'w') as f:
            json.dump(results, f)
        print(f'  [{len(results)}/{len(pairs)}] saved to Drive')

# Final save
with open(CACHE_PATH, 'w') as f:
    json.dump(results, f)
print(f'Done. Total processed: {len(results)}')

## Results

In [ ]:
ious = [r['iou'] for r in results]
detected = [r for r in results if r['detected']]
iou_det = [r['iou'] for r in detected]
n = len(results)

if n == 0:
    print('No results yet (run the evaluation cell first)')
else:
    print(f'IoU Evaluation Summary  (N={n})')
    print(f'Detection rate (>=1 box found): {len(detected)/n*100:.1f}%')
    print(f'Mean  IoU (all images): {np.mean(ious):.3f}')
    print(f'Median IoU (all images): {np.median(ious):.3f}')
    if iou_det:
        print(f'Mean  IoU (detected only): {np.mean(iou_det):.3f}')
    else:
        print('Mean  IoU (detected only): N/A')
    print(f'IoU >= 0.5 (good detections): {sum(v>=0.5 for v in ious)/n*100:.1f}%')
    print(f'Min / Max IoU: {np.min(ious):.3f} / {np.max(ious):.3f}')



In [ ]:
if not results:
    print('No results to plot (run the evaluation cell first)')
else:
    ious = [r['iou'] for r in results]
    plt.figure(figsize=(7, 4))
    plt.hist(ious, bins=20, range=(0, 1), color='steelblue', edgecolor='white')
    plt.axvline(np.mean(ious), color='red', linestyle='--', label=f'Mean IoU = {np.mean(ious):.3f}')
    plt.axvline(0.5, color='orange', linestyle='--', label='IoU = 0.5 threshold')
    plt.xlabel('IoU')
    plt.ylabel('Number of images')
    plt.title(f'IoU Score Distribution  (N={len(ious)} images)')
    plt.legend()
    plt.tight_layout()
    plt.show()
